In [1]:
import numpy as np
# ==========================================
# ML dataset parameter range
# ==========================================

W = 0.2
K = 0.2
BW = 1.024e-4

U_RESONANCE = (W + 1.0) / K
U_PHASE = W / K

# Initial parallel velocity
U_PARALLEL_MIN = 5.8
U_PARALLEL_MAX = 6.2

# Initial perpendicular velocity
U_PERP_MIN = 0.1
U_PERP_MAX = 1.0

# Initial relative phase
PSI_MIN = -np.pi
PSI_MAX = np.pi

In [4]:
# ==========================================
# Time resolution
# ==========================================

DT = 0.01

SAVE_EVERY = 10  #每10个 Boris 时间步保存一次数据

DT_SAVE = DT * SAVE_EVERY

TAU_END = 10000.0 # 时间步长扩大两倍

N_STEPS_ML = int(round(TAU_END / DT))
N_SAVED_ML = N_STEPS_ML // SAVE_EVERY + 1

print("Boris integration dt =", DT)
print("ML data interval =", DT_SAVE)
print("Total Boris steps =", N_STEPS_ML)
print("Saved time points =", N_SAVED_ML)

Boris integration dt = 0.01
ML data interval = 0.1
Total Boris steps = 1000000
Saved time points = 100001


In [3]:
# 使用随机均匀采样，选取2000个粒子用于机器学习数据集
# 这是随机均匀采样，不是 linspace
# ==========================================
# Initial conditions for ML dataset
# ==========================================

N_PARTICLES_ML = 2000

rng_ml = np.random.default_rng(42)

u_parallel0_ml = rng_ml.uniform(
    U_PARALLEL_MIN,
    U_PARALLEL_MAX,
    N_PARTICLES_ML
)

u_perp0_ml = rng_ml.uniform(
    U_PERP_MIN,
    U_PERP_MAX,
    N_PARTICLES_ML
)

psi0_ml = rng_ml.uniform(
    PSI_MIN,
    PSI_MAX,
    N_PARTICLES_ML
)

print("Number of ML particles =", N_PARTICLES_ML)

print(
    "u_parallel range =",
    u_parallel0_ml.min(),
    u_parallel0_ml.max()
)

print(
    "u_perp range =",
    u_perp0_ml.min(),
    u_perp0_ml.max()
)

print(
    "psi range =",
    psi0_ml.min(),
    psi0_ml.max()
)

Number of ML particles = 2000
u_parallel range = 5.800207497666282 6.199768763689676
u_perp range = 0.10057807332988636 0.9997888032229095
psi range = -3.139617096701777 3.1398010099778033


In [5]:
# ==========================================
# Convert initial conditions
# ==========================================

x0_ml = np.zeros(
    N_PARTICLES_ML,
    dtype=np.float64
)

u_x0_ml = u_parallel0_ml.copy()

u_y0_ml = (
    u_perp0_ml * np.sin(psi0_ml)
)

u_z0_ml = (
    u_perp0_ml * np.cos(psi0_ml)
)

tau0_ml = 0.0


# Check
u_perp_check = np.sqrt(
    u_y0_ml**2 + u_z0_ml**2
)

print(
    "Maximum u_perp reconstruction error =",
    np.max(
        np.abs(
            u_perp_check - u_perp0_ml
        )
    )
)

Maximum u_perp reconstruction error = 1.1102230246251565e-16


In [6]:
# ==========================================
# Prescribed wave fields
# ==========================================

def wave_phase(x, tau):
    return K * x - W * tau


def magnetic_field(x, tau):

    phi_B = wave_phase(x, tau)

    B_x = np.ones_like(x)
    B_y = BW * np.sin(phi_B)
    B_z = BW * np.cos(phi_B)

    return B_x, B_y, B_z


def electric_field(x, tau):

    phi_B = wave_phase(x, tau)

    E_x = np.zeros_like(x)

    E_y = (
        U_PHASE
        * BW
        * np.cos(phi_B)
    )

    E_z = (
        -U_PHASE
        * BW
        * np.sin(phi_B)
    )

    return E_x, E_y, E_z

In [7]:
# ==========================================
# Boris pusher
# ==========================================

def boris_push(
    x,
    u_x,
    u_y,
    u_z,
    tau,
    dt
):

    # 1. Half-step position
    x_half = (
        x + 0.5 * dt * u_x
    )

    tau_half = (
        tau + 0.5 * dt
    )

    # 2. Fields at midpoint
    E_x, E_y, E_z = electric_field(
        x_half,
        tau_half
    )

    B_x, B_y, B_z = magnetic_field(
        x_half,
        tau_half
    )

    E = np.column_stack(
        (E_x, E_y, E_z)
    )

    B = np.column_stack(
        (B_x, B_y, B_z)
    )

    u = np.column_stack(
        (u_x, u_y, u_z)
    )

    # 3. Half electric kick
    u_minus = (
        u + 0.5 * dt * E
    )

    # 4. Magnetic rotation
    t = 0.5 * dt * B

    t_squared = np.sum(
        t**2,
        axis=1
    )

    s = (
        2.0 * t
        / (1.0 + t_squared[:, None])
    )

    u_prime = (
        u_minus
        + np.cross(u_minus, t)
    )

    u_plus = (
        u_minus
        + np.cross(u_prime, s)
    )

    # 5. Second half electric kick
    u_new = (
        u_plus + 0.5 * dt * E
    )

    # 6. Second half-step position
    x_new = (
        x_half
        + 0.5 * dt * u_new[:, 0]
    )

    tau_new = tau + dt

    return (
        x_new,
        u_new[:, 0],
        u_new[:, 1],
        u_new[:, 2],
        tau_new
    )

In [8]:
# ==========================================
# ML history arrays
# ==========================================

u_parallel_history_ml = np.empty(
    (
        N_SAVED_ML,
        N_PARTICLES_ML
    ),
    dtype=np.float32
)

u_perp_history_ml = np.empty(
    (
        N_SAVED_ML,
        N_PARTICLES_ML
    ),
    dtype=np.float32
)

psi_history_ml = np.empty(
    (
        N_SAVED_ML,
        N_PARTICLES_ML
    ),
    dtype=np.float32
)

print(
    "History shape =",
    u_parallel_history_ml.shape
)

memory_gb = (
    u_parallel_history_ml.nbytes
    + u_perp_history_ml.nbytes
    + psi_history_ml.nbytes
) / 1024**3

print(
    "Total history memory =",
    memory_gb,
    "GB"
)

History shape = (100001, 2000)
Total history memory = 2.235196530818939 GB


In [9]:
# ==========================================
# Current particle states
# ==========================================

x_ml = x0_ml.copy()

u_x_ml = u_x0_ml.copy()
u_y_ml = u_y0_ml.copy()
u_z_ml = u_z0_ml.copy()

tau_ml = tau0_ml

In [10]:
# ==========================================
# Save initial state
# ==========================================

save_index = 0

# u_parallel
u_parallel_history_ml[save_index] = (
    u_x_ml
)

# u_perp
u_perp_history_ml[save_index] = np.sqrt(
    u_y_ml**2
    + u_z_ml**2
)

# Particle gyrophase Phi
phi_ml = np.arctan2(
    u_y_ml,
    u_z_ml
)

# Wave phase Phi_B
phi_B_ml = wave_phase(
    x_ml,
    tau_ml
)

# Relative phase
psi_ml = phi_ml - phi_B_ml

# Wrap to [-pi, pi)
psi_history_ml[save_index] = (
    psi_ml + np.pi
) % (2.0 * np.pi) - np.pi

save_index += 1

In [11]:
import time

# ==========================================
# Main ML trajectory simulation
# ==========================================

start_time = time.time()

for step in range(
    1,
    N_STEPS_ML + 1
):

    (
        x_ml,
        u_x_ml,
        u_y_ml,
        u_z_ml,
        tau_ml
    ) = boris_push(
        x_ml,
        u_x_ml,
        u_y_ml,
        u_z_ml,
        tau_ml,
        DT
    )

    # Save every SAVE_EVERY Boris steps
    if step % SAVE_EVERY == 0:

        # Parallel velocity
        u_parallel_history_ml[
            save_index
        ] = u_x_ml

        # Perpendicular velocity
        u_perp_history_ml[
            save_index
        ] = np.sqrt(
            u_y_ml**2
            + u_z_ml**2
        )

        # Particle gyrophase
        phi_ml = np.arctan2(
            u_y_ml,
            u_z_ml
        )

        # Wave phase
        phi_B_ml = wave_phase(
            x_ml,
            tau_ml
        )

        # Relative phase
        psi_ml = (
            phi_ml
            - phi_B_ml
        )

        # Wrap into [-pi, pi)
        psi_history_ml[
            save_index
        ] = (
            psi_ml + np.pi
        ) % (2.0 * np.pi) - np.pi

        save_index += 1

    # Progress output
    if step % 100000 == 0:

        elapsed = (
            time.time() - start_time
        )

        print(
            f"step = {step:,} / "
            f"{N_STEPS_ML:,}, "
            f"tau = {tau_ml:.1f}, "
            f"elapsed = {elapsed:.1f} s"
        )


elapsed = (
    time.time() - start_time
)

print()
print("Simulation finished.")
print("Final tau =", tau_ml)
print("Saved states =", save_index)
print(
    f"Total runtime = "
    f"{elapsed:.2f} s"
)

step = 100,000 / 1,000,000, tau = 1000.0, elapsed = 16.4 s
step = 200,000 / 1,000,000, tau = 2000.0, elapsed = 32.3 s
step = 300,000 / 1,000,000, tau = 3000.0, elapsed = 48.2 s
step = 400,000 / 1,000,000, tau = 4000.0, elapsed = 64.0 s
step = 500,000 / 1,000,000, tau = 5000.0, elapsed = 80.0 s
step = 600,000 / 1,000,000, tau = 6000.0, elapsed = 96.0 s
step = 700,000 / 1,000,000, tau = 7000.0, elapsed = 112.0 s
step = 800,000 / 1,000,000, tau = 8000.0, elapsed = 128.3 s
step = 900,000 / 1,000,000, tau = 9000.0, elapsed = 144.9 s
step = 1,000,000 / 1,000,000, tau = 10000.0, elapsed = 161.9 s

Simulation finished.
Final tau = 10000.000000171856
Saved states = 100001
Total runtime = 161.93 s


In [12]:
print(
    "u_parallel dtype =",
    u_parallel_history_ml.dtype
)

print(
    "u_perp dtype =",
    u_perp_history_ml.dtype
)

print(
    "psi dtype =",
    psi_history_ml.dtype
)

print(
    "Shape =",
    psi_history_ml.shape
)

u_parallel dtype = float32
u_perp dtype = float32
psi dtype = float32
Shape = (100001, 2000)


In [13]:
print(
    "NaN in u_parallel:",
    np.isnan(
        u_parallel_history_ml
    ).any()
)

print(
    "NaN in u_perp:",
    np.isnan(
        u_perp_history_ml
    ).any()
)

print(
    "NaN in psi:",
    np.isnan(
        psi_history_ml
    ).any()
)

NaN in u_parallel: False
NaN in u_perp: False
NaN in psi: False


In [14]:
from pathlib import Path

# ==========================================
# Data directory
# ==========================================

DATA_DIR = Path("data/ml_dataset")

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Data directory =", DATA_DIR.resolve())

np.savez(
    DATA_DIR / "ml_dataset.npz",

    u_parallel=u_parallel_history_ml,
    u_perp=u_perp_history_ml,
    psi=psi_history_ml,

    u_parallel0=u_parallel0_ml.astype(np.float32),
    u_perp0=u_perp0_ml.astype(np.float32),
    psi0=psi0_ml.astype(np.float32),

    W=W,
    K=K,
    BW=BW,

    DT=DT,
    SAVE_EVERY=SAVE_EVERY,
    DT_SAVE=DT_SAVE,
    TAU_END=TAU_END,

    N_PARTICLES_ML=N_PARTICLES_ML,
    N_STEPS_ML=N_STEPS_ML,
    N_SAVED_ML=N_SAVED_ML
)

Data directory = /Users/zerek/Desktop/plasma physics/AI_hoshino/data/ml_dataset


In [15]:
output_file = DATA_DIR / "ml_dataset.npz"

print("Saved file exists =", output_file.exists())
print("File path =", output_file.resolve())

Saved file exists = True
File path = /Users/zerek/Desktop/plasma physics/AI_hoshino/data/ml_dataset/ml_dataset.npz
